# Bài 3: Phân ngưỡng, Co và Giãn (Thresholding, Erosion, and Dilation)

Phân ngưỡng (thresholding) chuyển một ảnh xám thành ảnh nhị phân (đen/trắng) bằng cách so sánh từng điểm ảnh với một giá trị ngưỡng. Kết quả thường bị nhiễu, nên ta làm sạch nó bằng hai phép toán hình thái học (morphological operations) cơ bản: **co (erosion)** (thu nhỏ vùng trắng, loại bỏ các đốm nhỏ) và **giãn (dilation)** (mở rộng vùng trắng, lấp các lỗ nhỏ).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tạo một ảnh thử nghiệm có nhiễu

Chúng ta tổng hợp một ảnh xám có một hình sáng trên nền tối, rồi thêm nhiễu ngẫu nhiên để một số điểm ảnh nền bị sáng và một số điểm ảnh tiền cảnh bị tối.

In [ ]:
rng = np.random.default_rng(0)

gray = np.full((120, 120), 40, dtype=np.uint8)
cv2.circle(gray, (60, 60), 35, 220, -1)

noise = rng.normal(0, 35, gray.shape)
noisy = np.clip(gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# sprinkle a few salt-and-pepper specks
speckle_coords = rng.integers(0, 120, size=(60, 2))
for y, x in speckle_coords:
    noisy[y, x] = 255 if noisy[y, x] < 128 else 0

plt.imshow(noisy, cmap='gray', vmin=0, vmax=255)
plt.title('Noisy grayscale image')
plt.axis('off')
plt.show()

## Phân ngưỡng

Phân ngưỡng chẳng qua là một phép so sánh trên từng điểm ảnh: mọi điểm ảnh lớn hơn ngưỡng trở thành trắng (255), mọi điểm ảnh nhỏ hơn hoặc bằng ngưỡng trở thành đen (0). Với NumPy, đó chỉ là một phép so sánh boolean cộng với `np.where` để chọn giá trị đầu ra.

In [ ]:
threshold_value = 128
binary = (noisy > threshold_value) * np.uint8(255)
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)
plt.title(f'Thresholded (t={threshold_value})')
plt.axis('off')
plt.show()

### Hàm `cv2.threshold` của OpenCV

OpenCV gói cùng phép toán này vào `cv2.threshold`, rất tiện vì hàm còn hỗ trợ các biến thể ngoài phân ngưỡng nhị phân đơn giản &mdash; đầu ra đảo ngược, cắt trần thay vì đưa về 0, và tự động chọn ngưỡng (`cv2.THRESH_OTSU`) &mdash; tất cả qua cùng một hàm. Với phân ngưỡng nhị phân thông thường, nó tính ra kết quả giống hệt phiên bản NumPy ở trên.

In [ ]:
_, binary_cv2 = cv2.threshold(noisy, threshold_value, 255, cv2.THRESH_BINARY)

print('cv2.threshold matches the NumPy version exactly:', np.array_equal(binary, binary_cv2))

## Phương pháp Otsu: tự động chọn ngưỡng

Giá trị `threshold_value = 128` ở trên được chọn thủ công. **Phương pháp Otsu (Otsu's method)** (<a href="../references.html#otsu-1979">Otsu, 1979</a><span class="landmark-paper">&#9733;</span>) chọn ngưỡng một cách tự động: nó coi histogram của ảnh là hỗn hợp của hai lớp (tiền cảnh và nền) và duyệt qua mọi ngưỡng có thể để tìm ngưỡng làm cực tiểu phương sai *trong lớp* (within-class variance) (tương đương với cực đại phương sai *giữa* hai lớp) &mdash; tức ngưỡng chia histogram tốt nhất thành hai cụm chặt và tách biệt rõ. Phương pháp giả định histogram có dạng gần như hai đỉnh (bimodal), điều mà một hình sáng trên nền tối thỏa mãn khá tốt.

In [ ]:
thresh_otsu, binary_otsu = cv2.threshold(noisy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'Otsu-selected threshold: {thresh_otsu:.1f}   (we hand-picked {threshold_value} above)')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
axes[0].hist(noisy.ravel(), bins=50, color='gray')
axes[0].axvline(thresh_otsu, color='red', linestyle='--', label=f'Otsu t={thresh_otsu:.0f}')
axes[0].set_title('Histogram', fontsize=10)
axes[0].legend(fontsize=8)
axes[1].imshow(binary, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Manual (t={threshold_value})', fontsize=10)
axes[2].imshow(binary_otsu, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Otsu (t={thresh_otsu:.0f})', fontsize=10)
for ax in axes[1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

Ngưỡng do Otsu tự động tính ra ở đây khá gần với giá trị chọn thủ công. Otsu là lựa chọn mặc định tốt bất cứ khi nào cần một ngưỡng, nhưng hãy nhớ rằng nó sẽ không hoạt động với các histogram tách biệt kém hoặc có nhiều đỉnh.

## Phép co

Có hai **phép toán hình thái học** cơ bản (co và giãn) để làm sạch các đốm muối tiêu (salt-and-pepper) và các cạnh lởm chởm trong kết quả trên. **Phép co (erosion)** trượt một **phần tử cấu trúc (structuring element)** nhỏ (kernel) trên ảnh; một điểm ảnh chỉ giữ màu trắng nếu *toàn bộ* kernel nằm gọn trong vùng trắng. Điều này làm thu nhỏ vùng trắng và loại bỏ các đốm trắng nhỏ.

In [ ]:
kernel = np.ones((3, 3), np.uint8)
eroded = cv2.erode(binary, kernel, iterations=1)

plt.imshow(eroded, cmap='gray', vmin=0, vmax=255)
plt.title('Eroded')
plt.axis('off')
plt.show()

Trong ảnh động này, kernel 3x3 trượt qua mọi điểm ảnh tiền cảnh, và một điểm ảnh chỉ tồn tại nếu toàn bộ kernel rơi vào vùng trắng. Đốm cô lập và phần nhô ra mảnh rộng một điểm ảnh đều bị loại bỏ, còn khối đặc thì co vào trong một điểm ảnh ở mỗi phía.

<img src="../animations/erosion_demo.gif" alt="Ảnh động minh họa phép co nhị phân với kernel 3x3" style="display: block; margin: 0 auto; max-width: 650px;">

## Phép giãn

**Phép giãn (dilation)** làm điều ngược lại: một điểm ảnh trở thành trắng nếu kernel chồng lên *bất kỳ* phần nào của vùng trắng. Điều này làm mở rộng vùng trắng và lấp các lỗ/đốm đen nhỏ.

In [ ]:
dilated = cv2.dilate(binary, kernel, iterations=1)

plt.imshow(dilated, cmap='gray', vmin=0, vmax=255)
plt.title('Dilated')
plt.axis('off')
plt.show()

Trong ảnh động này, cùng kernel 3x3 đó trượt qua mọi điểm ảnh nền gần vùng trắng, và một điểm ảnh được tô trắng nếu kernel chồng lên bất kỳ điểm ảnh trắng nào. Một điểm ảnh đơn lẻ lớn thành một khối đặc, khoảng trống rộng một điểm ảnh giữa hai điểm ảnh được nối liền, và một đường mảnh trở nên dày hơn.

<img src="../animations/dilation_demo.gif" alt="Ảnh động minh họa phép giãn nhị phân với kernel 3x3" style="display: block; margin: 0 auto; max-width: 750px;">

## Phép mở: co rồi giãn

Áp dụng phép co rồi đến phép giãn (gọi là **phép mở (opening)**) giúp loại bỏ các đốm trắng nhỏ trong khi khôi phục kích thước của vùng chính — một cách phổ biến để khử nhiễu ảnh đã phân ngưỡng.

In [ ]:
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [binary, eroded, dilated, opened],
    ['Thresholded', 'Eroded', 'Dilated', 'Opened\n(erode then dilate)'],
):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Bài tập

1. Hãy thử `cv2.MORPH_CLOSE` (giãn rồi co) thay cho `MORPH_OPEN`. Nó khác thế nào, và nó khắc phục loại nhiễu điểm ảnh đen nào mà phép mở không làm được?
2. Tăng kích thước kernel lên `(5, 5)`. Kết quả thay đổi thế nào so với `(3, 3)`?